In [9]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path
import re
import xml.etree.ElementTree as ET
from functools import reduce
import openpyxl
from scipy.optimize import minimize_scalar
import numpy as np
import glob

pd.set_option('display.max_columns', None)

# Schooling

In [10]:
import requests
import pandas as pd
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

def fetch_worldbank_indicator(country="ua", indicator="SE.TER.ENRR", start_year=2000, end_year=2024):
    url = f"https://api.worldbank.org/v2/country/{country}/indicator/{indicator}"

    session = requests.Session()
    retry = Retry(
        total=5,
        connect=5,
        read=5,
        backoff_factor=1.5,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"]
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter)
    session.mount("http://", adapter)

    params = {
        "format": "json",
        "per_page": 200,
        "date": f"{start_year}:{end_year}"
    }

    resp = session.get(url, params=params, timeout=(10, 360))
    resp.raise_for_status()
    data = resp.json()

    rows = data[1]

    df = pd.DataFrame([
        {
            "country": row["country"]["value"],
            "year": int(row["date"]),
            "school_enrollment": row["value"]
        }
        for row in rows if row["value"] is not None
    ])

    return df.sort_values("year").reset_index(drop=True)

school_enrollment = fetch_worldbank_indicator()
school_enrollment

,country,year,school_enrollment
0,Ukraine,2000,50.507118
1,Ukraine,2001,52.910110
2,Ukraine,2002,56.645679
3,Ukraine,2003,60.255322
4,Ukraine,2004,63.440750
5,Ukraine,2005,66.889862
6,Ukraine,2006,71.939812
7,Ukraine,2007,76.262756
8,Ukraine,2008,79.732521
9,Ukraine,2009,82.499252


In [11]:
school_enrollment["quarter_period"] = pd.PeriodIndex(
    school_enrollment["year"].astype(str) + "Q4",
    freq="Q"
)

full_quarters = pd.period_range(
    start=f"{school_enrollment['year'].min()}Q1",
    end=f"{school_enrollment['year'].max()}Q4",
    freq="Q"
)

school_enrollment = (
    school_enrollment
    .set_index("quarter_period")[["school_enrollment"]]
    .reindex(full_quarters)
)

school_enrollment["school_enrollment_q_interp"] = (
    school_enrollment["school_enrollment"]
    .interpolate(method="linear", limit_direction="both")
)

school_enrollment = school_enrollment.reset_index().rename(columns={"index": "quarter_period"})
school_enrollment["year_quarter"] = school_enrollment["quarter_period"].astype(str)

school_enrollment = school_enrollment[school_enrollment['year_quarter']>='2010Q1'][["year_quarter", "school_enrollment_q_interp"]]

school_enrollment

,year_quarter,school_enrollment_q_interp
40,2010Q1,82.351502
41,2010Q2,82.203752
42,2010Q3,82.056001
43,2010Q4,81.908251
44,2011Q1,82.630715
45,2011Q2,83.353179
46,2011Q3,84.075642
47,2011Q4,84.798106
48,2012Q1,84.568554
49,2012Q2,84.339003


# Other indicator (global economy)

In [12]:
indicators = pd.read_csv("raw_csvs/21-04-26 03_47_45_theglobaleconomy.csv")
indicators = indicators[[
    'Year',
    'Rule of law index (-2.5 weak; 2.5 strong)',
    'Government effectiveness index (-2.5 weak; 2.5 strong)',
    'Corruption Perceptions Index 100 = no corruption',
    'Fiscal freedom index (0-100)', 'Business freedom index (0-100)',
    'Financial freedom index (0-100)',
    'Economic freedom overall index (0-100)'
]]
indicators

,Year,Rule of law index (-2.5 weak; 2.5 strong),Government effectiveness index (-2.5 weak; 2.5 strong),Corruption Perceptions Index 100 = no corruption,Fiscal freedom index (0-100),Business freedom index (0-100),Financial freedom index (0-100),Economic freedom overall index (0-100)
0,2010,-0.81,-0.81,24,78.0,39.0,30.0,46.0
1,2011,-0.77,-0.83,23,77.0,47.0,30.0,46.0
2,2012,-0.67,-0.70,26,78.0,46.0,30.0,46.0
3,2013,-0.72,-0.74,25,78.0,48.0,30.0,46.0
4,2014,-0.73,-0.52,26,79.0,60.0,30.0,49.0
5,2015,-0.70,-0.55,27,79.0,59.0,30.0,47.0
6,2016,-0.65,-0.52,29,79.0,57.0,30.0,47.0
7,2017,-0.58,-0.48,30,79.0,62.0,30.0,48.0
8,2018,-0.52,-0.36,32,80.0,63.0,30.0,52.0
9,2019,-0.59,-0.37,30,82.0,66.0,30.0,52.0


In [ ]:
df_q = indicators.loc[indicators.index.repeat(4)].reset_index(drop=True)

df_q["quarter"] = ["Q1", "Q2", "Q3", "Q4"] * len(indicators)
df_q["year_quarter"] = df_q["Year"].astype(str) + df_q["quarter"]

df_q = df_q.drop(columns="quarter")

cols = ["year_quarter"] + [c for c in df_q.columns if c != "year_quarter" and c != "Year"]
df_q = df_q[cols]

df_q

,year_quarter,Rule of law index (-2.5 weak; 2.5 strong),Government effectiveness index (-2.5 weak; 2.5 strong),Corruption Perceptions Index 100 = no corruption,Fiscal freedom index (0-100),Business freedom index (0-100),Financial freedom index (0-100),Economic freedom overall index (0-100)
0,2010Q1,-0.81,-0.81,24,78.0,39.0,30.0,46.0
1,2010Q2,-0.81,-0.81,24,78.0,39.0,30.0,46.0
2,2010Q3,-0.81,-0.81,24,78.0,39.0,30.0,46.0
3,2010Q4,-0.81,-0.81,24,78.0,39.0,30.0,46.0
4,2011Q1,-0.77,-0.83,23,77.0,47.0,30.0,46.0
5,2011Q2,-0.77,-0.83,23,77.0,47.0,30.0,46.0
6,2011Q3,-0.77,-0.83,23,77.0,47.0,30.0,46.0
7,2011Q4,-0.77,-0.83,23,77.0,47.0,30.0,46.0
8,2012Q1,-0.67,-0.70,26,78.0,46.0,30.0,46.0
9,2012Q2,-0.67,-0.70,26,78.0,46.0,30.0,46.0


# Final

In [14]:
final = school_enrollment.merge(df_q, on="year_quarter", how="inner")
final.to_csv("data/indicators_with_school_enrollment.csv", index=False)
final.head()

,year_quarter,school_enrollment_q_interp,Rule of law index (-2.5 weak; 2.5 strong),Government effectiveness index (-2.5 weak; 2.5 strong),Corruption Perceptions Index 100 = no corruption,Fiscal freedom index (0-100),Business freedom index (0-100),Financial freedom index (0-100),Economic freedom overall index (0-100)
0,2010Q1,82.351502,-0.81,-0.81,24,78.0,39.0,30.0,46.0
1,2010Q2,82.203752,-0.81,-0.81,24,78.0,39.0,30.0,46.0
2,2010Q3,82.056001,-0.81,-0.81,24,78.0,39.0,30.0,46.0
3,2010Q4,81.908251,-0.81,-0.81,24,78.0,39.0,30.0,46.0
4,2011Q1,82.630715,-0.77,-0.83,23,77.0,47.0,30.0,46.0
